Scarping free patterns from https://www.crochet.com/patterns/free

In [75]:
import requests
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
import cloudscraper

In [89]:
def scrape_product(product_link):
    scraper = cloudscraper.create_scraper()

    r = scraper.get(product_link)
    soup = BeautifulSoup(r.text, "html.parser")

    row = {}

    button = None
    for b in soup.find_all("button"):
        if "Pattern Description" in b.get_text():
            button = b
            desc_div = button.find_parent("h3").find_next_sibling("div")
            row["description"] = desc_div.get_text(" ", strip=True)
        
        if "Pattern Details" in b.get_text():
            button_details = b
            details_div = button_details.find_parent("h3").find_next_sibling("div")
            items = details_div.find_all("li")

            for li in items:
                label = li.find("b")

                if label:
                    key = label.get_text(strip=True).replace(":", "")
                    value = label.next_sibling.strip()

                    row[key] = value
        
    return row

In [90]:
url = "https://www.crochet.com/filet-crochet-market-bag/p/58066"

scrape_product(url)

{'description': "Make a stunning market bag for trips to the farmers' market, outings to the beach, or to carry a few project bags worth of crochet goodies. You'll want to take this customizable bag everywhere! Practice the filet crochet technique with this mesh bag. Create patterns in the shape of a heart, a flower, a chevron pattern, or create a chart all your own. You will be a filet crochet expert by the time you finish this fast and fun market bag. Grab a project bundle and save!",
 'Pattern Type': 'Crochet',
 'Difficulty': 'Intermediate',
 'Sizes Included': '27" circumference, 15" long',
 'Yarn Weight': 'DK',
 'Yarn Line': '',
 'Yardage': '436',
 'Fiber Type': 'Cotton',
 'Needle / Hook Sizes': 'Size G (4.0mm) Crochet hook'}

In [92]:
base_url = "https://www.crochet.com/patterns/free"
rows = []
page = 1

scraper = cloudscraper.create_scraper()

while True:
    print(f"Scraping page {page}...")

    if page == 1:
        url = base_url
    else:
        url = f"{base_url}?Free-Paid=Free&page={page}"

    if page == 8:
        break

    response = scraper.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.select("div.group.relative")
    if not cards:
        print("No more products found. Ending scrape.")
        break

    for card in cards:
        link = card.select_one("a[href]")
        product_link = base_url + link["href"] 

        img_tag = card.select_one("img")
        image_link = img_tag["src"] if img_tag else None

        name_tag = card.select_one("h3 a")
        name = name_tag.get_text(strip=True) if name_tag else None
        
        product_data = scrape_product(product_link)

        product_data["name"] = name
        product_data["pattern_link"] = product_link
        product_data["image_link"] = image_link

        rows.append(product_data)
        time.sleep(1)

    page += 1

df = pd.DataFrame(rows)

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...


In [95]:
df.head()

,name,pattern_link,image_link
0,Wavy Chevron Dishcloth,https://www.crochet.com/patterns/free/wavy-che...,https://d2q9kw5vp0we94.cloudfront.net/i/w=1000...
1,Filet Crochet Market Bag,https://www.crochet.com/patterns/free/filet-cr...,https://d2q9kw5vp0we94.cloudfront.net/i/w=1000...
2,Good Earth Blanket,https://www.crochet.com/patterns/free/good-ear...,https://d2q9kw5vp0we94.cloudfront.net/i/w=1000...
3,Aberdeen Pullover,https://www.crochet.com/patterns/free/aberdeen...,https://d2q9kw5vp0we94.cloudfront.net/i/w=1000...
4,Adventure Time Blanket,https://www.crochet.com/patterns/free/adventur...,https://d2q9kw5vp0we94.cloudfront.net/i/w=1000...


In [79]:
import cloudscraper
from bs4 import BeautifulSoup

url = "https://www.crochet.com/filet-crochet-market-bag/p/58066"
scraper = cloudscraper.create_scraper()

r = scraper.get(url)

print(r.status_code)

soup = BeautifulSoup(r.text, "html.parser")

print(len(soup.text))

button_desc = soup.find("button", string="Pattern Description")
print(button_desc)

200
12818
None


In [86]:
soup.find_all("button")

[<button aria-expanded="false" class="rt-popout__trigger cursor-pointer lg:hidden" id="offcanvas-nav-drawer-trigger" type="button"><svg aria-labelledby="lucide-menu-title-69b6e4946a09a" class="lucide lucide-menu w-[22px] h-[22px] icon__menu" fill="none" height="24" role="img" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="24" xmlns="http://www.w3.org/2000/svg"><title id="lucide-menu-title-69b6e4946a09a">Menu Icon</title><line x1="4" x2="20" y1="12" y2="12"></line><line x1="4" x2="20" y1="6" y2="6"></line><line x1="4" x2="20" y1="18" y2="18"></line></svg></button>,
 <button aria-expanded="false" class="rt-popout__trigger rt-desktop-nav-menu-trigger z-50 flex items-center h-full uppercase text-(--header-nav-text) border-b-2 border-b-transparent border-t-2 border-t-transparent hover:border-b-brand hover:text-(--header-nav-text-active) px-3" id="id779921981" type="button">
                     Yarn
                 </button>

In [88]:
for b in soup.find_all("button"):
    if "Pattern Description" in b.get_text():
        button = b
        break
print(button)

<button aria-controls="disclosure-1" aria-expanded="false" class="rt-toggler__trigger rt-pdp-features__trigger group relative flex w-full items-center justify-between py-6 text-left rt-toggler__trigger--open" id="id-3042531e27" type="button">
<!-- Open: "text-indigo-600", Closed: "text-gray-900" -->
                        Pattern Description
                    <svg aria-labelledby="lucide-plus-title-69b6e49471b94" class="lucide lucide-plus w-[20px] h-[20px] text-gray-700 icon__plus" fill="none" height="24" role="img" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" viewbox="0 0 24 24" width="24" xmlns="http://www.w3.org/2000/svg"><title id="lucide-plus-title-69b6e49471b94">Plus Icon</title><path d="M5 12h14"></path><path d="M12 5v14"></path></svg>
<svg aria-labelledby="lucide-minus-title-69b6e49471ba4" class="lucide lucide-minus w-[20px] h-[20px] text-gray-700 icon__minus" fill="none" height="24" role="img" stroke="currentColor" stroke-linecap="ro